In [ ]:
########################5-1 실습########################
# 라이브러리 설치
# 모델 및 토크나이저 로드

def calculate_full_ft_memory(model):
    """Full Fine-tuning에 필요한 예상 메모리를 계산합니다."""
    # TODO: 총 파라미터 수를 계산하세요 (힌트: sum(p.numel() for p in model.parameters()))
    total_params = sum(p.numel() for p in model.parameters())
    
    # FP16 기준 Full FT 메모리 계산
    # 모델 가중치: 2 bytes per param (FP16)
    # Gradient: 2 bytes per param (FP16)
    # Optimizer (AdamW): 8 bytes per param (FP32 optimizer states)
    # Activations: 모델 × 2 (보수적 추정)
    
    # TODO: 각 구성요소별 메모리를 GB 단위로 계산하세요
    model_memory = (total_params * 2) / (1024**3)
    grad_memory = (total_params * 2) / (1024**3)
    optim_memory = (total_params * 8) / (1024**3)
    activation_memory = model_memory * 2
    
    # TODO: 총 필요 메모리를 계산하세요
    total_memory = model_memory + grad_memory + optim_memory + activation_memory
    
    return total_params, {
        'model': model_memory,
        'gradient': grad_memory, 
        'optimizer': optim_memory,
        'activation': activation_memory,
        'total': total_memory
    }



In [ ]:
# TODO: LoRA 설정값 채우기
"""LoRA 설정값 채우기"""
# TODO: rank 값을 설정하세요 (8, 16, 32, 64 중 선택)
r = 8

# TODO: r * 2 권장
lora_alpha = 16

# TODO: LoRA를 적용할 레이어 리스트를 설정하세요
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# TODO: 0을 권장
lora_dropout = 0.0

bias = "none"

# LoRA 적용
model = FastModel.get_peft_model(
    model,
    r=r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias=bias,
)

print(f"LoRA 적용 완료!")
print(f"- r (rank): {r}")
print(f"- lora_alpha: {lora_alpha}")
print(f"- target_modules: {target_modules}")

In [ ]:
# TODO: convert_to_chatml 함수 작성
# LLM 파인튜닝을 위해 데이터를 system/user/assistant 역할 형식으로 변환 

# TODO: conversations 리스트를 완성하세요
# 각 딕셔너리는 "role"과 "content" 키를 가져야 합니다
# role: "system", "user", "assistant" 중 하나
# content: example의 적절한 필드 값
def convert_to_chatml(example):
    """데이터셋을 chat 형식으로 변환합니다."""
    return {
        "conversations": [
            # TODO: system, user, assistant 역할의 메시지를 작성하세요
            {"role": "system", "content": example["task"]},
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["expected_output"]}
        ]
    }

# 데이터셋 변환
dataset = dataset.map(convert_to_chatml)
print(f"변환된 샘플: {dataset[0]['conversations']}")



In [ ]:
### TODO 4: 추론을 위한 instruction/response part 채우기
# TODO: 추론 테스트 (학습에 사용하지 않은 테스트 데이터 활용)

# 테스트 입력 구성 (Step 1에서 저장한 test_dataset 사용)
# 학습에 사용하지 않은 데이터로 테스트해야 공정한 평가 가능!
test_sample = test_dataset[0]  # 첫 번째 테스트 샘플

# TODO: 테스트 입력 구성 - system과 user 역할의 메시지 리스트를 만드세요
# 힌트: {"role": "system", "content": ...}, {"role": "user", "content": ...}
test_messages = [
    {"role": "system", "content": test_sample["task"]},
    {"role": "user", "content": test_sample["input"]}
    # TODO: system 역할의 메시지 추가
    # TODO: user 역할의 메시지 추가
]

# Chat template 적용
text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,  # 추론 시 True로 설정
).removeprefix('<bos>')

print("입력 텍스트:")
print(text[:500] + "...")